# Quora Duplicate Questions — EDA & Baseline Model

## 1. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import scipy.sparse as sp
from collections import Counter
from datasets import load_dataset
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

## 2. Data Loading

In [ ]:
ds = load_dataset(
    "sentence-transformers/quora-duplicates",
    "pair-class"
)
df = ds['train'].to_pandas()
print(f"Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 3. EDA — Basic Inspection

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

## 4. Noise & Missing Value Analysis

In [ ]:
print("Null values per column:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
# Drop rows where either sentence is null
df.dropna(subset=['sentence1', 'sentence2'], inplace=True)

# Drop fully duplicate rows
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Shape after cleaning: {df.shape}")

## 5. Label Distribution

In [ ]:
df['label'].value_counts().plot(kind='bar', color=['steelblue', 'coral'])
plt.xlabel("Label (0 = Non-Duplicate, 1 = Duplicate)")
plt.ylabel("Count")
plt.title("Duplicate vs Non-Duplicate Questions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(df['label'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

## 6. Text Length Analysis

In [ ]:
df['que1_length'] = df['sentence1'].apply(len)
df['que2_length'] = df['sentence2'].apply(len)
df[['que1_length', 'que2_length']].describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['que1_length'], bins=30, alpha=0.7, label='Sentence 1')
plt.hist(df['que2_length'], bins=30, alpha=0.7, label='Sentence 2')
plt.title("Character Length Distribution")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df['que1_words'] = df['sentence1'].apply(lambda x: len(x.split()))
df['que2_words'] = df['sentence2'].apply(lambda x: len(x.split()))
print(df[['que1_words', 'que2_words']].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(df['que1_words'], ax=axes[0])
axes[0].set_title("Word Count Outliers — Sentence 1")
sns.boxplot(df['que2_words'], ax=axes[1])
axes[1].set_title("Word Count Outliers — Sentence 2")
plt.tight_layout()
plt.show()

## 7. NLP Pipeline

### 7a. Tokenization

In [ ]:
# Tokenize using NLTK word_tokenize
df['tokens_q1'] = df['sentence1'].apply(lambda x: word_tokenize(x.lower()))
df['tokens_q2'] = df['sentence2'].apply(lambda x: word_tokenize(x.lower()))

print("Sample tokens (sentence1):")
print(df['tokens_q1'][0])
print("\nSample tokens (sentence2):")
print(df['tokens_q2'][0])

### 7b. Most Common Words

In [ ]:
stop_words = set(stopwords.words('english'))

# Flatten all tokens from sentence1, remove stopwords and punctuation
all_tokens = [
    token for tokens in df['tokens_q1']
    for token in tokens
    if token.isalpha() and token not in stop_words
]

common_words = Counter(all_tokens).most_common(20)
words = [i[0] for i in common_words]
counts = [i[1] for i in common_words]

plt.figure(figsize=(12, 5))
plt.bar(words, counts, color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.title("Top 20 Most Common Words (sentence1, stopwords removed)")
plt.tight_layout()
plt.show()

### 7c. Text Cleaning

In [ ]:
def clean_text(text):
    """Lowercase and remove special characters from text."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text

df['clean_q1'] = df['sentence1'].apply(clean_text)
df['clean_q2'] = df['sentence2'].apply(clean_text)

df[['sentence1', 'clean_q1', 'sentence2', 'clean_q2']].head()

## 8. Sample Analysis — Duplicate & Non-Duplicate Questions

In [ ]:
duplicate_questions = df[df['label'] == 1]
print(f"Total duplicate pairs: {len(duplicate_questions):,}")
duplicate_questions[['sentence1', 'sentence2']].head()

In [ ]:
non_duplicate = df[df['label'] == 0]
print(f"Total non-duplicate pairs: {len(non_duplicate):,}")
non_duplicate[['sentence1', 'sentence2']].head()

## 9. Baseline Model — TF-IDF + Logistic Regression

### 9a. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

q1_tfidf = tfidf.fit_transform(df['clean_q1'])
q2_tfidf = tfidf.transform(df['clean_q2'])

X = sp.hstack([q1_tfidf, q2_tfidf])
y = df['label']

print(f"Feature matrix shape: {X.shape}")

### 9b. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}")

### 9c. Model Training

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Model training complete.")

### 9d. Evaluation

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("F1 Score     :", f1_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

### 9e. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Dup', 'Duplicate'],
            yticklabels=['Non-Dup', 'Duplicate'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

### 9f. Sample Predictions

In [ ]:
results = df.iloc[y_test.index].copy()
results['Predicted'] = y_pred
results[['clean_q1', 'clean_q2', 'label', 'Predicted']].head(10)